# Task 3: AQE Experiment (Queries 3, 5)

Compare the same Gold-table queries with Adaptive Query Execution (AQE) disabled and enabled.

## Connect to Spark

In [1]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query, register_integrated_taxi_trips

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-AQE-Queries3-5')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 10:35:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Configure the experiment

Only `spark.sql.adaptive.enabled` changes between the two runs. AQE uses runtime statistics to re-optimize a plan.

In [2]:
QUERY_NAMES = (
    'query_03_air_quality_vs_demand',
    'query_05_peak_travel_hours',
)
REPETITIONS = 3

AQE_DISABLED = 'false'
AQE_ENABLED = 'true'

## Measure AQE disabled and enabled

Each mode receives one warm-up followed by three timed runs.

In [3]:
def measure_aqe_mode(enabled):
    spark.catalog.clearCache()
    spark.conf.set('spark.sql.adaptive.enabled', enabled)
    register_integrated_taxi_trips(spark)
    return {
        query_name: measure_registered_query(spark, query_name, REPETITIONS)
        for query_name in QUERY_NAMES
    }

aqe_disabled_results = measure_aqe_mode(AQE_DISABLED)
aqe_enabled_results = measure_aqe_mode(AQE_ENABLED)

for query_name in QUERY_NAMES:
    assert rows_match(
        aqe_disabled_results[query_name]['rows'],
        aqe_enabled_results[query_name]['rows'],
    ), f'AQE results differ for {query_name}.'

print('AQE-enabled and AQE-disabled results match within float tolerance.')

26/09/19 10:35:32 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/19 10:35:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


AQE-enabled and AQE-disabled results match within float tolerance.


## AQE-disabled physical plans

In [4]:
spark.conf.set('spark.sql.adaptive.enabled', AQE_DISABLED)
register_integrated_taxi_trips(spark)

for query_name in QUERY_NAMES:
    print(f'AQE-disabled plan: {query_name}')
    execute_registered_query(spark, query_name).explain('formatted')

AQE-disabled plan: query_03_air_quality_vs_demand
== Physical Plan ==
* Sort (16)
+- Exchange (15)
   +- * BroadcastNestedLoopJoin Cross BuildRight (14)
      :- * HashAggregate (7)
      :  +- Exchange (6)
      :     +- * HashAggregate (5)
      :        +- * Project (4)
      :           +- * Filter (3)
      :              +- * ColumnarToRow (2)
      :                 +- Scan parquet  (1)
      +- BroadcastExchange (13)
         +- * HashAggregate (12)
            +- Exchange (11)
               +- * HashAggregate (10)
                  +- * HashAggregate (9)
                     +- ReusedExchange (8)


(1) Scan parquet 
Output [3]: [pu_datetime_hour#5616, average_value_pu#5633, date#5647]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
PushedFilters: [IsNotNull(average_value_pu)]
ReadSchema: struct<pu_datetime_hour:timestamp,average_value_pu:double>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [pu_datetime_hour#5616, average_val

## AQE-enabled physical plans

AEQ enabled -> `AdaptiveSparkPlan` and changed shuffles or joins.

In [5]:
spark.conf.set('spark.sql.adaptive.enabled', AQE_ENABLED)
register_integrated_taxi_trips(spark)

for query_name in QUERY_NAMES:
    print(f'AQE-enabled plan: {query_name}')
    query = execute_registered_query(spark, query_name)
    query.collect()
    query.explain('formatted')

AQE-enabled plan: query_03_air_quality_vs_demand
== Physical Plan ==
AdaptiveSparkPlan (44)
+- == Final Plan ==
   ResultQueryStage (25)
   +- * Sort (24)
      +- AQEShuffleRead (23)
         +- ShuffleQueryStage (22), Statistics(sizeInBytes=102.4 KiB, rowCount=2.18E+3)
            +- Exchange (21)
               +- * BroadcastNestedLoopJoin Cross BuildRight (20)
                  :- * HashAggregate (9)
                  :  +- AQEShuffleRead (8)
                  :     +- ShuffleQueryStage (7), Statistics(sizeInBytes=93.9 KiB, rowCount=2.41E+3)
                  :        +- Exchange (6)
                  :           +- * HashAggregate (5)
                  :              +- * Project (4)
                  :                 +- * Filter (3)
                  :                    +- * ColumnarToRow (2)
                  :                       +- Scan parquet  (1)
                  +- BroadcastQueryStage (19), Statistics(sizeInBytes=24.0 B, rowCount=1)
                     +- BroadcastEx

## Benchmark summary

In [6]:
summary = []
for query_name in QUERY_NAMES:
    disabled = aqe_disabled_results[query_name]
    enabled = aqe_enabled_results[query_name]
    summary.append({
        'query': query_name,
        'aqe_disabled_samples_seconds': [
            round(value, 3) for value in disabled['samples_seconds']
        ],
        'aqe_disabled_median_seconds': round(disabled['median_seconds'], 3),
        'aqe_enabled_samples_seconds': [
            round(value, 3) for value in enabled['samples_seconds']
        ],
        'aqe_enabled_median_seconds': round(enabled['median_seconds'], 3),
        'aqe_speedup_factor': round(
            disabled['median_seconds'] / enabled['median_seconds'],
            2,
        ),
    })
summary

[{'query': 'query_03_air_quality_vs_demand',
  'aqe_disabled_samples_seconds': [1.073, 1.518, 0.788],
  'aqe_disabled_median_seconds': 1.073,
  'aqe_enabled_samples_seconds': [0.572, 0.505, 0.484],
  'aqe_enabled_median_seconds': 0.505,
  'aqe_speedup_factor': 2.13},
 {'query': 'query_05_peak_travel_hours',
  'aqe_disabled_samples_seconds': [0.785, 0.754, 0.752],
  'aqe_disabled_median_seconds': 0.754,
  'aqe_enabled_samples_seconds': [0.702, 0.679, 0.715],
  'aqe_enabled_median_seconds': 0.702,
  'aqe_speedup_factor': 1.07}]

## Cleanup

In [7]:
spark.catalog.clearCache()
spark.stop()